# Day 34：RAG 评估体系：Faithfulness、答案相关性、上下文精确率/召回率与类 Ragas 框架（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-034` ｜ **开发分支**：`feat/day34-ragas-eval-framework`  
> **配套 HTML5 课件**：`courseware/day34/index.html` ｜ **独立版本快照**：`versions/day34/`

---

## 🎙️ 老陈开场旁白

早啊！老板昨天在评审会上的追问，恰恰是**区分「玩具 Demo」和「生产级 AI 工程」的分水岭**——**没有量化评估，一切优化都是玄学！**
传统软件靠单元测试断言 `assert a == b`，但 RAG 生成的是自然语言，你怎么用自动化脚本评估它好不好？业界最通用的黄金标准就是 **Ragas（RAG Assessment）四维评估三角**：
- **检得准不准（检索层）**：`Context Precision`（上下文精确率：相关文档是不是排在最前面？） + `Context Recall`（上下文召回率：标准答案里的关键事实是不是都被搜回来了？）；
- **答得对不对（生成层）**：`Faithfulness`（忠实度：模型回答里的每一句话是不是都能在检索上下文里找到出处，有没有瞎编数字？） + `Answer Relevancy`（答案相关性：有没有答非所问？）。
今天老陈带大家亲手把这 4 个指标的数学逻辑全部写透，并做出一套可以塞进 GitLab CI 卡点的 `MiniRagasEvaluator` 自动化评测框架！

### 承上启下
- **昨日基线**：昨天（Day 33）我们上线了「BM25 + 向量 + RRF + Reranker + 父文档」超级检索引擎。但在评审会上，老板直接灵魂发问：「你们说新版本比 Naive RAG 强很多，到底强了多少？幻觉率降到多少了？有没有数据报告？上线后万一改了个 Prompt 把旧功能改坏了怎么兜底？」
- **明日预告**：明天（Day 35）我们将探索另一个企业级 RAG 顶流框架——LlamaIndex，拆解其 Document/Node/Index/QueryEngine 架构，并与 LangChain 路线做全面基准对比选型！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day34'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-034）

星辰智脑 RAG 引擎即将进入项目二全量开发，团队需要一套自动化、可重复执行的 RAG 评估工具，对每次切片策略、Prompt、模型或检索器变更进行量化回归测试。

**今日迭代目标：**
- 实现 Ragas 核心四维评估指标：`compute_faithfulness`、`compute_answer_relevancy`、`compute_context_precision`、`compute_context_recall`。
- 实现 `MiniRagasEvaluator` 批量评测引擎：支持单样本诊断（精准定位「幻觉 / 漏召回 / 排序差 / 答非所问」）、CI 门禁校验与 Markdown 对比报告生成。
- 对比评测 `v1_naive_rag` 与 `v2_hybrid_rerank_rag` 两套系统在黄金测试集上的四维得分。

**验收标准：**
- 包含编造数字（如上下文写 600 元、回答写 800 元）的幻觉样本，`Faithfulness` 得分必须显著低于忠实样本（< 0.6）。
- 相关上下文排在 Top-1 的样本，`Context Precision` 必须严格高于排在末尾的样本。
- CI 门禁能准确拦截不达标版本（`gate_passed == False`），放行达标版本（`gate_passed == True`）。
- 所有 pytest 自动化测试 100% 本地离线通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 四维指标出问题时，如何秒级定位根因？

老陈给大家总结一张**排障速查表**：
- `Context Recall` 低 -> **召回阶段问题**：检查切分是否把答案切碎了、Embedding 是否不适合中文、是不是该加 BM25 混合检索或 Multi-Query。
- `Context Recall` 高但 `Context Precision` 低 -> **排序阶段问题**：正确答案搜回来了但沉底了，快上 Day 33 的 `bge-reranker`！
- `Context Recall` 高但 `Faithfulness` 低 -> **生成阶段幻觉**：Prompt 约束太弱、Temperature 调太高，或者底座模型指令遵循能力不足。
- `Faithfulness` 高但 `Answer Relevancy` 低 -> **答非所问**：模型把上下文照抄了一通但没回答用户真正关心的点，需优化生成 Prompt 与上下文压缩。

> **⚠️ 老陈避坑**：踩坑提醒：不要只看四维总平均分！如果 `Faithfulness` 是 0.50（疯狂瞎编）而另外三项都是 1.0，平均分也有 0.875，看似很美实则线上事故。CI 门禁必须对**每个维度单独设底线**！

### 2. 2. 生产环境官方 Ragas 框架标准用法

生产环境中，Ragas 使用高能力大模型作为裁判（LLM-as-a-Judge）来执行断言拆解与 NLI 蕴含判断：
```python
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

result = evaluate(
    dataset=Dataset.from_list(samples),
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=bge_embeddings,
)
print(result.to_pandas())
```

> **⚠️ 老陈避坑**：踩坑提醒：LLM-as-a-Judge 的裁判模型最好选能力强于或不同于生成模型的模型（如 DeepSeek-V3 / Qwen-Max / GPT-4o），并将裁判温度固定为 `temperature=0` 以保证评测结果可复现。


## 💻 实操步骤 1：步骤一：实现 Ragas 四维核心评估算法（Faithfulness / Relevancy / Precision / Recall）

- **目标文件**：`versions/day34/src/day34_rag_metrics.py`
- **老陈代码旁白**：咱们把 Ragas 最核心的 4 个评估指标用清晰透明的算法完整实现出来，特别加入了「关键数字一致性核查」，专治大模型把 600 元篡改成 800 元的隐蔽幻觉！


In [ ]:
"""
Day 34 - 步骤一：类 Ragas 四维核心评估指标实现
包含：Faithfulness、Answer Relevancy、Context Precision (AP@K)、Context Recall
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List, Sequence, Set


@dataclass
class EvalSample:
    """单条 RAG 评测样本（与 Ragas Dataset 字段对齐）。"""

    sample_id: str
    question: str
    answer: str
    contexts: List[str]
    ground_truth: str


def extract_keywords(text: str) -> Set[str]:
    """提取用于事实核查的核心词元（中文二元词 + 英文/错误码/数字）。"""
    lowered = text.lower()
    tokens: Set[str] = set()
    # 英文词、错误码与数字
    tokens.update(re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", lowered))
    # 中文二元词组（过滤极常见虚词组合）
    stop_bigrams = {"的是", "可以", "需要", "进行", "相关", "以及", "一个", "什么", "怎么", "多少", "请问", "根据"}
    for chunk in re.findall(r"[\u4e00-\u9fff]+", lowered):
        for i in range(len(chunk) - 1):
            bg = chunk[i : i + 2]
            if bg not in stop_bigrams:
                tokens.add(bg)
    return tokens


def extract_numbers(text: str) -> Set[str]:
    """提取文本中的所有关键数字（如 600、15、5042、3），用于严查数字幻觉。"""
    return set(re.findall(r"\d+(?:\.\d+)?", text))


def split_claims(text: str) -> List[str]:
    """将一段自然语言回答拆解为独立的原子事实断言（Claims）。"""
    raw = [s.strip() for s in re.split(r"[。！？；;\n，,]", text) if len(s.strip()) >= 4]
    return raw if raw else ([text.strip()] if text.strip() else [])


def compute_faithfulness(answer: str, contexts: Sequence[str]) -> float:
    """
    忠实度（Faithfulness）：
    衡量模型回答中的所有原子断言是否都能在检索上下文中找到事实依据。
    若某条断言包含上下文中不存在的关键数字（如把 600 编成 800），直接判定为幻觉不支持！
    """
    claims = split_claims(answer)
    if not claims or not contexts:
        return 0.0

    merged_ctx = "\n".join(contexts)
    ctx_tokens = extract_keywords(merged_ctx)
    ctx_numbers = extract_numbers(merged_ctx)

    supported = 0
    for claim in claims:
        c_nums = extract_numbers(claim)
        # 严格数字幻觉检查：断言里提到的数字必须全部出现在检索上下文中
        if c_nums and not c_nums.issubset(ctx_numbers):
            continue
        c_tokens = extract_keywords(claim)
        if not c_tokens:
            supported += 1
            continue
        overlap_ratio = len(c_tokens & ctx_tokens) / len(c_tokens)
        if overlap_ratio >= 0.45:
            supported += 1

    return round(supported / len(claims), 4)


def compute_answer_relevancy(question: str, answer: str) -> float:
    """
    答案相关性（Answer Relevancy）：
    衡量回答是否切中提问核心主题，并对拒答/套话/答非所问施加惩罚。
    """
    if not question.strip() or not answer.strip():
        return 0.0
    evasive_phrases = ("不知道", "无法回答", "抱歉", "与您的问题无关", "天气不错")
    if any(p in answer for p in evasive_phrases):
        return 0.1

    # 去除疑问词后再提取核心意图词，避免「是多少/怎么」拉低正常陈述回答得分
    cleaned_q = re.sub(r"请问|是多少|多少|几天|怎么|如何|什么|是否|吗|呢|需要|报错", " ", question)
    q_tokens = extract_keywords(cleaned_q)
    a_tokens = extract_keywords(answer)
    if not q_tokens:
        return 0.0

    # 问题关键意图在回答中的覆盖度
    intent_coverage = len(q_tokens & a_tokens) / len(q_tokens)
    # 适度考虑回答信息密度（过短或严重跑题都会拉低分数）
    length_factor = min(1.0, len(answer.strip()) / 12.0)
    score = min(1.0, (0.85 * intent_coverage + 0.15 * length_factor))
    return round(score, 4)


def compute_context_precision(
    question: str,
    contexts: Sequence[str],
    ground_truth: str,
) -> float:
    """
    上下文精确率（Context Precision，采用 AP@K 位置加权公式）：
    不仅看检索回来的上下文是否相关，更奖励将相关上下文排在靠前位置（Top-1 > Top-3）。
    """
    if not contexts:
        return 0.0

    ref_tokens = extract_keywords(question) | extract_keywords(ground_truth)
    gt_nums = extract_numbers(ground_truth)

    relevances: List[int] = []
    for ctx in contexts:
        c_tokens = extract_keywords(ctx)
        c_nums = extract_numbers(ctx)
        overlap = len(c_tokens & ref_tokens) / max(1, len(ref_tokens))
        num_match = bool(gt_nums and (gt_nums & c_nums))
        is_rel = 1 if (overlap >= 0.28 or num_match) else 0
        relevances.append(is_rel)

    total_rel = sum(relevances)
    if total_rel == 0:
        return 0.0

    ap_sum = 0.0
    running_rel = 0
    for rank, rel in enumerate(relevances, start=1):
        if rel == 1:
            running_rel += 1
            ap_sum += running_rel / rank
    return round(ap_sum / total_rel, 4)


def compute_context_recall(ground_truth: str, contexts: Sequence[str]) -> float:
    """
    上下文召回率（Context Recall）：
    将标准答案（Ground Truth）拆解为关键事实句，统计有多少比例已被检索上下文覆盖。
    """
    gt_claims = split_claims(ground_truth)
    if not gt_claims or not contexts:
        return 0.0

    merged_ctx = "\n".join(contexts)
    ctx_tokens = extract_keywords(merged_ctx)
    ctx_numbers = extract_numbers(merged_ctx)

    recalled = 0
    for claim in gt_claims:
        c_nums = extract_numbers(claim)
        if c_nums and not c_nums.issubset(ctx_numbers):
            continue
        c_tokens = extract_keywords(claim)
        if not c_tokens:
            recalled += 1
            continue
        if (len(c_tokens & ctx_tokens) / len(c_tokens)) >= 0.45:
            recalled += 1
    return round(recalled / len(gt_claims), 4)


if __name__ == "__main__":
    ctx_good = [
        "一线城市（北京、上海、深圳、广州）出差住宿上限为每晚600元，需提前3天在OA审批。",
        "园区地下车库免费向员工开放。",
    ]
    q = "北京出差住宿上限是多少，需要提前几天OA审批？"
    gt = "北京出差住宿上限每晚600元，需提前3天OA审批。"
    ans_faithful = "北京属于一线城市，出差住宿上限每晚600元，需提前3天在OA审批。"
    ans_hallucinated = "北京出差住宿上限每晚800元，需提前3天在OA审批。"

    print("忠实回答 Faithfulness :", compute_faithfulness(ans_faithful, ctx_good))
    print("幻觉回答 Faithfulness :", compute_faithfulness(ans_hallucinated, ctx_good))
    print("相关上下文排第1 Precision:", compute_context_precision(q, ctx_good, gt))
    print("相关上下文排第2 Precision:", compute_context_precision(q, list(reversed(ctx_good)), gt))
    print("Context Recall        :", compute_context_recall(gt, ctx_good))


### 🔍 步骤 1 代码深度解析

1. `compute_faithfulness` 专门增加了 `extract_numbers` 数字一致性强校验：只要模型把上下文里的「600元」瞎编成「800元」，该断言立刻判死刑，绝不放水！
2. `compute_context_precision` 严格遵循 AP@K 公式，把同样一条相关文档从第 1 位挪到第 2 位，分数会从 `1.0` 直接腰斩到 `0.5`，完美体现 Rerank 排序质量。

**预期运行输出参考：**
```text
忠实回答 Faithfulness : 1.0
幻觉回答 Faithfulness : 0.5
相关上下文排第1 Precision: 1.0
相关上下文排第2 Precision: 0.5
Context Recall        : 1.0
```


## 💻 实操步骤 2：步骤二：实现类 Ragas 自动化评测框架（MiniRagasEvaluator）与 CI 质量门禁报告

- **目标文件**：`versions/day34/src/day34_ragas_runner.py`
- **老陈代码旁白**：单个指标算清楚了，现在把它封装成工程级的 `MiniRagasEvaluator`：批量评测整个测试集、给每条 Bad Case 自动开诊断处方、校验 CI 质量门禁，并输出可以直接贴进 GitLab MR 的 Markdown 对比报告！


In [ ]:
"""
Day 34 - 步骤二：类 Ragas 自动化评测框架、Bad Case 根因诊断与 CI 门禁报告
"""

from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Sequence

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day34_rag_metrics import (
    EvalSample,
    compute_answer_relevancy,
    compute_context_precision,
    compute_context_recall,
    compute_faithfulness,
)


@dataclass
class SampleEvalResult:
    sample_id: str
    question: str
    faithfulness: float
    answer_relevancy: float
    context_precision: float
    context_recall: float
    diagnosis: List[str] = field(default_factory=list)


@dataclass
class EvalReport:
    version_name: str
    sample_results: List[SampleEvalResult]
    summary: Dict[str, float]
    gate_passed: bool
    failed_gates: List[str]


class MiniRagasEvaluator:
    """企业级 RAG 自动化评测与 CI 门禁引擎。"""

    def __init__(
        self,
        min_faithfulness: float = 0.85,
        min_answer_relevancy: float = 0.70,
        min_context_precision: float = 0.75,
        min_context_recall: float = 0.80,
    ) -> None:
        self.thresholds = {
            "faithfulness": min_faithfulness,
            "answer_relevancy": min_answer_relevancy,
            "context_precision": min_context_precision,
            "context_recall": min_context_recall,
        }

    def evaluate_dataset(self, version_name: str, samples: Sequence[EvalSample]) -> EvalReport:
        results: List[SampleEvalResult] = []
        for s in samples:
            f_score = compute_faithfulness(s.answer, s.contexts)
            r_score = compute_answer_relevancy(s.question, s.answer)
            p_score = compute_context_precision(s.question, s.contexts, s.ground_truth)
            c_score = compute_context_recall(s.ground_truth, s.contexts)

            diag: List[str] = []
            if c_score < 0.7:
                diag.append("检索漏召回(需优化切分/混合检索)")
            if c_score >= 0.7 and p_score < 0.7:
                diag.append("排序靠后(需开启bge-reranker精排)")
            if f_score < 0.8:
                diag.append("生成幻觉(需收紧Prompt或核查数字约束)")
            if r_score < 0.6:
                diag.append("答非所问(需优化意图对齐)")
            if not diag:
                diag.append("PASS")

            results.append(
                SampleEvalResult(
                    sample_id=s.sample_id,
                    question=s.question,
                    faithfulness=f_score,
                    answer_relevancy=r_score,
                    context_precision=p_score,
                    context_recall=c_score,
                    diagnosis=diag,
                )
            )

        n = max(1, len(results))
        summary = {
            "faithfulness": round(sum(r.faithfulness for r in results) / n, 4),
            "answer_relevancy": round(sum(r.answer_relevancy for r in results) / n, 4),
            "context_precision": round(sum(r.context_precision for r in results) / n, 4),
            "context_recall": round(sum(r.context_recall for r in results) / n, 4),
        }

        failed_gates = [
            f"{metric}={summary[metric]:.4f} < 门禁阈值 {limit:.2f}"
            for metric, limit in self.thresholds.items()
            if summary[metric] < limit
        ]
        return EvalReport(
            version_name=version_name,
            sample_results=results,
            summary=summary,
            gate_passed=(len(failed_gates) == 0),
            failed_gates=failed_gates,
        )

    @staticmethod
    def render_markdown_comparison(report_a: EvalReport, report_b: EvalReport) -> str:
        lines = [
            "## StarMind RAG 自动化评测对比报告",
            f"| 评估维度 | {report_a.version_name} | {report_b.version_name} | 提升幅度 |",
            "| :--- | :---: | :---: | :---: |",
        ]
        for key in ("faithfulness", "answer_relevancy", "context_precision", "context_recall"):
            va = report_a.summary[key]
            vb = report_b.summary[key]
            delta = vb - va
            sign = "+" if delta >= 0 else ""
            lines.append(f"| `{key}` | {va:.4f} | {vb:.4f} | {sign}{delta:.4f} |")
        lines.append(
            f"\n- **{report_a.version_name} CI 门禁**: {'✅ PASS' if report_a.gate_passed else '❌ FAIL (' + '; '.join(report_a.failed_gates) + ')'}"
        )
        lines.append(
            f"- **{report_b.version_name} CI 门禁**: {'✅ PASS' if report_b.gate_passed else '❌ FAIL (' + '; '.join(report_b.failed_gates) + ')'}"
        )
        return "\n".join(lines)


def build_demo_eval_datasets() -> Dict[str, List[EvalSample]]:
    """构造 v1 Naive RAG 与 v2 Hybrid+Rerank RAG 的对照评测样本集。"""
    v1_naive = [
        EvalSample(
            sample_id="Q1",
            question="北京出差住宿上限是多少，需要提前几天OA审批？",
            answer="北京出差住宿上限每晚800元，需提前3天OA审批。",  # 存在数字幻觉
            contexts=[
                "园区班车每天早8点从地铁站发车。",
                "一线城市（北京、上海、深圳、广州）出差住宿上限为每晚600元，需提前3天在OA审批。",
            ],
            ground_truth="北京出差住宿上限每晚600元，需提前3天OA审批。",
        ),
        EvalSample(
            sample_id="Q2",
            question="ERR-5042 报错怎么排查？",
            answer="建议先重启电脑并重新连接公司 Wi-Fi 试试。",  # 漏召回导致瞎答
            contexts=[
                "员工入职当天请前往行政前台领取工牌。",
                "食堂二楼提供杭帮菜和轻食套餐。",
            ],
            ground_truth="错误码ERR-5042为网关超时，需检查上游显存水位并调大超时阈值。",
        ),
    ]

    v2_hybrid_rerank = [
        EvalSample(
            sample_id="Q1",
            question="北京出差住宿上限是多少，需要提前几天OA审批？",
            answer="北京属于一线城市，出差住宿上限每晚600元，需提前3天在OA审批。",
            contexts=[
                "一线城市（北京、上海、深圳、广州）出差住宿上限为每晚600元，需提前3天在OA审批。",
                "餐饮补助按每天100元定额随工资发放。",
            ],
            ground_truth="北京出差住宿上限每晚600元，需提前3天OA审批。",
        ),
        EvalSample(
            sample_id="Q2",
            question="ERR-5042 报错怎么排查？",
            answer="错误码ERR-5042表示网关超时，排查需检查上游显存水位并调大超时阈值。",
            contexts=[
                "错误码ERR-5042表示统一网关调用上游超时，排查需检查上游显存水位并调大超时阈值。",
                "错误码ERR-4019表示零信任VPN动态令牌失效。",
            ],
            ground_truth="错误码ERR-5042为网关超时，需检查上游显存水位并调大超时阈值。",
        ),
    ]
    return {"v1_naive_rag": v1_naive, "v2_hybrid_rerank": v2_hybrid_rerank}


if __name__ == "__main__":
    datasets = build_demo_eval_datasets()
    evaluator = MiniRagasEvaluator()
    rep_v1 = evaluator.evaluate_dataset("v1_naive_rag", datasets["v1_naive_rag"])
    rep_v2 = evaluator.evaluate_dataset("v2_hybrid_rerank", datasets["v2_hybrid_rerank"])
    print(evaluator.render_markdown_comparison(rep_v1, rep_v2))


### 🔍 步骤 2 代码深度解析

1. `MiniRagasEvaluator` 不仅给出 4 个宏观平均分，还会针对每条样本自动输出根因诊断标签（例如「检索漏召回」vs「排序靠后」vs「生成幻觉」），大幅缩短研发排障时间。
2. `render_markdown_comparison` 直接输出标准 Markdown 表格，实战中通过 GitLab CI 脚本自动评论到 Merge Request 下方，让 Code Review 有据可依。

**预期运行输出参考：**
```text
## StarMind RAG 自动化评测对比报告
| 评估维度 | v1_naive_rag | v2_hybrid_rerank | 提升幅度 |
| :--- | :---: | :---: | :---: |
| `faithfulness` | 0.2500 | 1.0000 | +0.7500 |
| `answer_relevancy` | 0.5200 | 0.9600 | +0.4400 |
| `context_precision` | 0.2500 | 1.0000 | +0.7500 |
| `context_recall` | 0.5000 | 1.0000 | +0.5000 |

- **v1_naive_rag CI 门禁**: ❌ FAIL (...)
- **v2_hybrid_rerank CI 门禁**: ✅ PASS
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day34/tests/test_day34.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 34 自动化单元测试：验证 Ragas 四维指标计算、幻觉捕获与 CI 门禁逻辑。"""

from src.day34_rag_metrics import (
    compute_answer_relevancy,
    compute_context_precision,
    compute_context_recall,
    compute_faithfulness,
)
from src.day34_ragas_runner import MiniRagasEvaluator, build_demo_eval_datasets


def test_faithfulness_detects_number_hallucination():
    ctx = ["一线城市出差酒店住宿上限为每晚600元，需提前3天OA申请。"]
    good_ans = "一线城市出差酒店住宿上限每晚600元，需提前3天OA申请。"
    bad_ans = "一线城市出差酒店住宿上限每晚800元，需提前3天OA申请。"

    f_good = compute_faithfulness(good_ans, ctx)
    f_bad = compute_faithfulness(bad_ans, ctx)
    assert f_good == 1.0
    assert f_bad < 0.6


def test_context_precision_rewards_top_ranking():
    q = "北京出差住宿上限是多少？"
    gt = "北京出差住宿上限每晚600元。"
    rel_ctx = "一线城市（北京、上海）出差住宿上限每晚600元。"
    noise_ctx = "公司每周五下午三点提供下午茶水果。"

    p_top1 = compute_context_precision(q, [rel_ctx, noise_ctx], gt)
    p_top2 = compute_context_precision(q, [noise_ctx, rel_ctx], gt)
    assert p_top1 == 1.0
    assert p_top2 == 0.5
    assert p_top1 > p_top2


def test_context_recall_and_relevancy():
    gt = "错误码ERR-5042表示网关超时，需检查显存水位。"
    ctx_full = ["错误码ERR-5042表示统一网关超时，请第一时间检查上游显存水位。"]
    ctx_miss = ["园区班车发车时间为早8点。"]

    assert compute_context_recall(gt, ctx_full) == 1.0
    assert compute_context_recall(gt, ctx_miss) == 0.0
    assert compute_answer_relevancy("ERR-5042 怎么排查", "抱歉，我不知道。") <= 0.2


def test_mini_ragas_ci_gate_blocks_v1_and_passes_v2():
    datasets = build_demo_eval_datasets()
    evaluator = MiniRagasEvaluator(
        min_faithfulness=0.85,
        min_answer_relevancy=0.70,
        min_context_precision=0.75,
        min_context_recall=0.80,
    )
    rep_v1 = evaluator.evaluate_dataset("v1_naive_rag", datasets["v1_naive_rag"])
    rep_v2 = evaluator.evaluate_dataset("v2_hybrid_rerank", datasets["v2_hybrid_rerank"])

    assert rep_v1.gate_passed is False
    assert len(rep_v1.failed_gates) >= 1
    assert rep_v2.gate_passed is True
    assert rep_v2.summary["faithfulness"] > rep_v1.summary["faithfulness"]

    md = evaluator.render_markdown_comparison(rep_v1, rep_v2)
    assert "v1_naive_rag" in md and "v2_hybrid_rerank" in md

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现 Answer Correctness（综合答案正确性 F1）指标

**题目要求：**
对比模型生成的 `answer` 与人工标注的 `ground_truth`，计算关键词集合的 Precision、Recall 与调和平均 F1（`compute_answer_correctness(answer, ground_truth)`），并验证完全正确、部分遗漏与答非所问三种场景。

**思路提示：** 提取两者词元集合 `A` 和 `G`，`precision = |A&G|/|A|`，`recall = |A&G|/|G|`，`f1 = 2*P*R/(P+R)`。


In [ ]:
"""Day 34 作业一参考答案：Answer Correctness 综合正确性 F1 指标"""
import re
from typing import Set


def _tokens(text: str) -> Set[str]:
    res = set(re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", text.lower()))
    for chunk in re.findall(r"[\u4e00-\u9fff]+", text):
        for i in range(len(chunk) - 1):
            res.add(chunk[i : i + 2])
    return res


def compute_answer_correctness(answer: str, ground_truth: str) -> float:
    a_set, g_set = _tokens(answer), _tokens(ground_truth)
    if not a_set or not g_set:
        return 0.0
    inter = len(a_set & g_set)
    p = inter / len(a_set)
    r = inter / len(g_set)
    return round(2 * p * r / (p + r), 4) if (p + r) > 0 else 0.0


if __name__ == "__main__":
    gt = "北京出差住宿上限每晚600元，需提前3天OA申请。"
    s1 = compute_answer_correctness("北京出差住宿上限每晚600元，需提前3天OA申请。", gt)
    s2 = compute_answer_correctness("北京出差住宿上限每晚600元。", gt)
    s3 = compute_answer_correctness("今天杭州西湖天气不错。", gt)
    print(f"完全匹配={s1}, 部分匹配={s2}, 无关回答={s3}")
    assert s1 == 1.0 and s1 > s2 > s3


**答案解析与评分标准：**

Answer Correctness 是端到端最终结果的直接标尺，与 Faithfulness 互补：前者看「对不对标准答案」，后者看「有没有背叛检索上下文」。


### 作业 2（企业实战）：作业二：实现回归退化检测器（Regression Guard）

**题目要求：**
输入基线版本指标字典 `baseline_summary` 与候选版本指标字典 `candidate_summary`，只要任意一个维度下降超过 `max_drop=0.02`（2 个百分点），就返回 `blocked=True` 及具体退化维度明细列表，防止新改动引发隐蔽退化。

**思路提示：** 遍历 4 个维度，计算 `drop = baseline[k] - candidate[k]`，若 `drop > max_drop` 则记录告警。


In [ ]:
"""Day 34 作业二参考答案：CI 回归退化守卫（Regression Guard）"""
from typing import Dict, List, Tuple


def check_regression(
    baseline: Dict[str, float],
    candidate: Dict[str, float],
    max_drop: float = 0.02,
) -> Tuple[bool, List[str]]:
    alerts: List[str] = []
    for metric, base_val in baseline.items():
        cand_val = candidate.get(metric, 0.0)
        drop = round(base_val - cand_val, 4)
        if drop > max_drop:
            alerts.append(f"{metric} 退化 {drop:.4f} (baseline={base_val} -> cand={cand_val})")
    return (len(alerts) > 0, alerts)


if __name__ == "__main__":
    base = {"faithfulness": 0.92, "context_recall": 0.90}
    cand_bad = {"faithfulness": 0.88, "context_recall": 0.93}
    blocked, msgs = check_regression(base, cand_bad, max_drop=0.02)
    print("是否阻断合并:", blocked, "告警:", msgs)
    assert blocked is True and "faithfulness" in msgs[0]


**答案解析与评分标准：**

很多线上 AI 事故都是「修好了 A 指标，悄悄把 B 指标弄崩了」；把 Regression Guard 写进 GitLab CI 是高成熟度团队的标配。


### 作业 3（架构进阶）：作业三：实现评测结果 JSONL 审计日志归档与趋势追踪

**题目要求：**
编写 `append_eval_audit(log_path, version, summary)` 与 `load_metric_trend(log_path, metric)`，将每次 CI 评测结果以 JSONL 格式追加落盘，并能按时间顺序提取指定指标的历史演化曲线数据。

**思路提示：** 使用 `json.dumps(..., ensure_ascii=False)` 逐行追加写入临时文件，再逐行 `json.loads` 读取提取指定字段。


In [ ]:
"""Day 34 作业三参考答案：评测审计日志 JSONL 归档与趋势读取"""
import json
import tempfile
from pathlib import Path
from typing import Dict, List, Tuple


def append_eval_audit(log_path: Path, version: str, summary: Dict[str, float]) -> None:
    record = {"version": version, "summary": summary}
    with log_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_metric_trend(log_path: Path, metric: str) -> List[Tuple[str, float]]:
    trend: List[Tuple[str, float]] = []
    for line in log_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        trend.append((obj["version"], float(obj["summary"][metric])))
    return trend


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp) / "rag_eval_audit.jsonl"
        append_eval_audit(p, "v1.0", {"faithfulness": 0.65, "context_recall": 0.70})
        append_eval_audit(p, "v1.1", {"faithfulness": 0.86, "context_recall": 0.85})
        append_eval_audit(p, "v2.0", {"faithfulness": 0.95, "context_recall": 0.94})
        trend = load_metric_trend(p, "faithfulness")
        print("Faithfulness 迭代趋势:", trend)
        assert trend[-1][1] > trend[0][1]


**答案解析与评分标准：**

将每次迭代的评测结果以不可变 JSONL 持久化，后续即可轻松接入 Grafana / ECharts 画出月度质量演进看板。


---

## 🎯 老陈收尾复盘

今天这套 `MiniRagasEvaluator` 一跑出来，老板看 v1 Naive RAG 和 v2 Hybrid+Rerank RAG 的四维雷达对比表，直接拍板通过！记住老陈一句话：**上线前不跑评测集、不设 Faithfulness 质量门禁的 RAG 系统，就是在线上裸奔。**明天咱们去看看 LlamaIndex 的世界！
